## 1. Đọc dữ liệu

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, month, sum as _sum, count, max as _max, avg, min as _min, datediff, to_timestamp

BASE_PATH = "/Lab04_DS/"

df_customers = spark.read.csv(BASE_PATH + "Customer_List.csv", header=True, inferSchema=True, sep=";")
df_orders = spark.read.csv(BASE_PATH + "Orders.csv", header=True, inferSchema=True, sep=";")
df_products = spark.read.csv(BASE_PATH + "Products.csv", header=True, inferSchema=True, sep=";")
df_order_items = spark.read.csv(BASE_PATH + "Order_Items.csv", header=True, inferSchema=True, sep=";")
df_order_reviews = spark.read.csv(BASE_PATH + "Order_Reviews.csv", header=True, inferSchema=True, sep=";")


## 2. Thống kê tổng số đơn hàng, số lượng khách hàng và người bán

In [2]:
total_orders = df_orders.count()
total_customers = df_customers.select("Customer_Trx_ID").distinct().count()
total_sellers = df_order_items.select("Seller_ID").distinct().count()

print("Tổng số đơn hàng:", total_orders)
print("Tổng số lượng khách hàng:", total_customers)
print("Tổng số lượng người bán:", total_sellers)


[Stage 19:>                                                         (0 + 1) / 1]

Tổng số đơn hàng: 99441
Tổng số lượng khách hàng: 99442
Tổng số lượng người bán: 3095


## 3. Phân tích số lượng đơn hàng theo quốc gia, sắp xếp theo thứ tự giảm dần

In [3]:
df_orders_cust = df_orders.join(df_customers, "Customer_Trx_ID", "inner")


df_orders_by_country = df_orders_cust.groupBy("Customer_Country") \
    .agg(count("Order_ID").alias("Tong_Don_Hang")) \
    .orderBy(col("Tong_Don_Hang").desc())

df_orders_by_country.show(truncate=False)


[Stage 26:>                                                         (0 + 1) / 1]

+----------------+-------------+
|Customer_Country|Tong_Don_Hang|
+----------------+-------------+
|Germany         |41754        |
|France          |12848        |
|Netherlands     |11629        |
|Belgium         |5464         |
|Austria         |5043         |
|Switzerland     |3640         |
|United Kingdom  |3382         |
|Poland          |2139         |
|Czechia         |2034         |
|Italy           |2025         |
|Spain           |1651         |
|Portugal        |1336         |
|Sweden          |975          |
|Denmark         |905          |
|Serbia          |746          |
|Norway          |716          |
|Slovakia        |534          |
|Slovenia        |495          |
|Turkey          |485          |
|Greece          |412          |
+----------------+-------------+
only showing top 20 rows



## 4. Phân tích số lượng đơn hàng nhóm theo năm, tháng đặt hàng (Hiển thị theo năm tăng dần, tháng giảm dần)

In [4]:
df_orders_timeline = df_orders.withColumn("Purchase_Date", to_timestamp(col("Order_Purchase_Timestamp"))) \
    .withColumn("Nam", year(col("Purchase_Date"))) \
    .withColumn("Thang", month(col("Purchase_Date")))


df_orders_timeline = df_orders_timeline.filter(col("Nam").isNotNull() & col("Thang").isNotNull())

df_orders_by_month = df_orders_timeline.groupBy("Nam", "Thang") \
    .agg(count("Order_ID").alias("So_Luong_Don_Hang")) \
    .orderBy(col("Nam").asc(), col("Thang").desc())

df_orders_by_month.show(truncate=False)


+----+-----+-----------------+
|Nam |Thang|So_Luong_Don_Hang|
+----+-----+-----------------+
|2022|12   |1                |
|2022|10   |324              |
|2022|9    |4                |
|2023|12   |5673             |
|2023|11   |7544             |
|2023|10   |4631             |
|2023|9    |4285             |
|2023|8    |4331             |
|2023|7    |4026             |
|2023|6    |3245             |
|2023|5    |3700             |
|2023|4    |2404             |
|2023|3    |2682             |
|2023|2    |1780             |
|2023|1    |800              |
|2024|10   |4                |
|2024|9    |16               |
|2024|8    |6512             |
|2024|7    |6292             |
|2024|6    |6167             |
+----+-----+-----------------+
only showing top 20 rows



## 5. Thống kê điểm đánh giá trung bình, số lượng đánh giá theo từng mức (ví dụ: 1 đến 5)
Loại bỏ các giá trị NULL và ngoại lệ trong cột Review_Score trước khi thực hiện.

In [5]:
df_valid_reviews = df_order_reviews.filter(
    col("Review_Score").isNotNull() & 
    (col("Review_Score") >= 1) & 
    (col("Review_Score") <= 5)
)


avg_score = df_valid_reviews.select(avg("Review_Score").alias("Diem_Trung_Binh"))
print("Điểm trung bình hệ thống:")
avg_score.show()


df_score_distribution = df_valid_reviews.groupBy("Review_Score") \
    .agg(count("Review_ID").alias("So_Luong_Danh_Gia")) \
    .orderBy(col("Review_Score").asc())

print("Phân phối đánh giá theo điểm:")
df_score_distribution.show()


Điểm trung bình hệ thống:


+------------------+
|   Diem_Trung_Binh|
+------------------+
|4.0864214950162765|
+------------------+

Phân phối đánh giá theo điểm:


[Stage 35:>                                                         (0 + 1) / 1]

+------------+-----------------+
|Review_Score|So_Luong_Danh_Gia|
+------------+-----------------+
|           1|            11424|
|           2|             3151|
|           3|             8179|
|           4|            19141|
|           5|            57328|
+------------+-----------------+



## 6. Tính doanh thu (giá sản phẩm + phí vận chuyển) trong năm 2024 và nhóm theo danh mục sản phẩm

In [6]:
df_orders_2024 = df_orders_timeline.filter(col("Nam") == 2024)

df_join_2024 = df_orders_2024.join(df_order_items, "Order_ID")

df_revenue_2024 = df_join_2024.join(df_products, "Product_ID")

df_revenue_2024 = df_revenue_2024.withColumn("Gia_Tri_Don", col("Price") + col("Freight_Value"))

df_revenue_by_cat = df_revenue_2024.groupBy("Product_Category_Name") \
    .agg(_sum("Gia_Tri_Don").alias("Tong_Doanh_Thu_2024")) \
    .orderBy(col("Tong_Doanh_Thu_2024").desc())

df_revenue_by_cat.show(truncate=False)


[Stage 40:>                                                         (0 + 1) / 1]

+-------------------------------+-------------------+
|Product_Category_Name          |Tong_Doanh_Thu_2024|
+-------------------------------+-------------------+
|Health_Beauty                  |885191.1199999922  |
|Watches_Gifts                  |771986.7500000049  |
|Bed_Bath_Table                 |650794.6999999998  |
|Sports_Leisure                 |621999.3399999999  |
|Computers_Accessories          |594771.0400000007  |
|Housewares                     |491576.96          |
|Furniture_Decor                |476466.13000000216 |
|Auto                           |404210.57000000094 |
|Baby                           |299052.56000000006 |
|Cool_Stuff                     |273910.05000000016 |
|Garden_Tools                   |259068.319999999   |
|Telephony                      |217452.129999999   |
|Perfumery                      |204562.54000000004 |
|Toys                           |200634.06999999995 |
|Office_Furniture               |181745.73000000004 |
|Stationery                 

## 7. Xác định sản phẩm có số lượng bán ra cao nhất và tính điểm đánh giá trung bình cho từng sản phẩm

In [7]:
df_qty_sold = df_order_items.groupBy("Product_ID") \
    .agg(count("Order_Item_ID").alias("So_Luong_Ban_Ra"))

df_product_reviews = df_order_reviews.join(df_order_items, "Order_ID")
df_avg_score = df_product_reviews.filter(col("Review_Score").isNotNull()) \
    .groupBy("Product_ID") \
    .agg(avg("Review_Score").alias("Diem_Trung_Binh_SanPham"))

df_product_stats = df_products.join(df_qty_sold, "Product_ID", "left") \
    .join(df_avg_score, "Product_ID", "left")

print("Danh sách top sản phẩm bán chạy nhất kèm đánh giá:")
df_product_stats.select("Product_ID", "Product_Category_Name", "So_Luong_Ban_Ra", "Diem_Trung_Binh_SanPham") \
    .orderBy(col("So_Luong_Ban_Ra").desc()) \
    .show(10, truncate=False)


Danh sách top sản phẩm bán chạy nhất kèm đánh giá:


+--------------------------------+---------------------+---------------+-----------------------+
|Product_ID                      |Product_Category_Name|So_Luong_Ban_Ra|Diem_Trung_Binh_SanPham|
+--------------------------------+---------------------+---------------+-----------------------+
|aca2eb7d00ea1a7b8ebd4e68314663af|Furniture_Decor      |527            |4.019083969465649      |
|99a4788cb24856965c36a24e339b6058|Bed_Bath_Table       |488            |3.8983402489626555     |
|422879e10f46682990de24d770e7f83d|Garden_Tools         |484            |3.9465020576131686     |
|389d119b48cf3043d311335e499d9c6b|Garden_Tools         |392            |4.117647058823529      |
|368c6c730842d78016ad823897a372db|Garden_Tools         |388            |3.922680412371134      |
|53759a2ecddad2bb87a079a1f1519f73|Garden_Tools         |373            |3.868632707774799      |
|d1c427060a0f73f6b889a5c7c61f2ac4|Computers_Accessories|343            |4.194117647058824      |
|53b36df67ebb7c41585e8d54d6772

## 8. Tính toán hiệu số giữa ngày giao hàng thực tế và ngày giao hàng dự kiến để đánh giá hiệu suất giao hàng

In [8]:
df_shipping = df_orders.join(df_order_items, "Order_ID")

df_delivery_perf = df_shipping.withColumn(
    "Ngay_Thuc_Te", to_timestamp(col("Order_Delivered_Carrier_Date"))
).withColumn(
    "Ngay_Du_Kien", to_timestamp(col("Shipping_Limit_Date"))
)

df_delivery_perf = df_delivery_perf.withColumn(
    "Do_Tre_Ngay", datediff(col("Ngay_Thuc_Te"), col("Ngay_Du_Kien"))
)

print("Số ngày trễ trung bình của các đơn hàng (Do_Tre_Ngay > 0 là giao trễ):")
df_delivery_perf.select(avg("Do_Tre_Ngay").alias("Ngay_Giao_Tre_TB")).show()


Số ngày trễ trung bình của các đơn hàng (Do_Tre_Ngay > 0 là giao trễ):


[Stage 52:>                                                         (0 + 1) / 1]

+-------------------+
|   Ngay_Giao_Tre_TB|
+-------------------+
|-3.4358580964685617|
+-------------------+



## 9. Nhóm khách hàng dựa trên số lượng đơn hàng, giá trị trung bình của đơn hàng và tần suất mua sắm

In [9]:
df_order_values = df_order_items.withColumn("Item_Total", col("Price") + col("Freight_Value")) \
    .groupBy("Order_ID") \
    .agg(_sum("Item_Total").alias("Order_Total_Value"))

df_customer_orders = df_orders_timeline.join(df_order_values, "Order_ID")

df_customer_stats = df_customer_orders.groupBy("Customer_Trx_ID") \
    .agg(
        count("Order_ID").alias("So_Luong_Don"),
        avg("Order_Total_Value").alias("Gia_Tri_TB_Don"),
        _max("Purchase_Date").alias("Don_Gan_Nhat"),
        _min("Purchase_Date").alias("Don_Xa_Nhat")
    )

df_customer_stats = df_customer_stats.withColumn(
    "Tan_Suat_Ngay", 
    datediff(col("Don_Gan_Nhat"), col("Don_Xa_Nhat")) / col("So_Luong_Don")
)

df_customer_stats.show(5)


[Stage 58:>                                                         (0 + 1) / 1]

+--------------------+------------+------------------+-------------------+-------------------+-------------+
|     Customer_Trx_ID|So_Luong_Don|    Gia_Tri_TB_Don|       Don_Gan_Nhat|        Don_Xa_Nhat|Tan_Suat_Ngay|
+--------------------+------------+------------------+-------------------+-------------------+-------------+
|f54a9f0e6b351c431...|           1|             35.95|2023-01-23 18:29:00|2023-01-23 18:29:00|          0.0|
|2a1dfb647f32f4390...|           1|            508.17|2024-06-01 12:23:00|2024-06-01 12:23:00|          0.0|
|4f28355e5c17a4a42...|           1|            153.72|2023-05-18 13:55:00|2023-05-18 13:55:00|          0.0|
|4632eb5a8f175f6fe...|           1| 91.66000000000001|2023-11-30 22:02:00|2023-11-30 22:02:00|          0.0|
|843ff05b30ce4f75b...|           1|102.74000000000001|2023-11-13 10:07:00|2023-11-13 10:07:00|          0.0|
+--------------------+------------+------------------+-------------------+-------------------+-------------+
only showing top 5 

## 10. Xếp hạng sellers dựa trên tổng doanh thu và số lượng đơn hàng bán được.

In [10]:
df_seller_items = df_order_items.withColumn("Item_Rev", col("Price") + col("Freight_Value"))

df_seller_stats = df_seller_items.groupBy("Seller_ID") \
    .agg(
        _sum("Item_Rev").alias("Tong_Doanh_Thu_Seller"),
        count("Order_Item_ID").alias("So_Don_Da_Ban")
    ) \
    .orderBy(col("Tong_Doanh_Thu_Seller").desc(), col("So_Don_Da_Ban").desc())

df_seller_stats.show(10, truncate=False)


[Stage 61:>                                                         (0 + 1) / 1]

+--------------------------------+---------------------+-------------+
|Seller_ID                       |Tong_Doanh_Thu_Seller|So_Don_Da_Ban|
+--------------------------------+---------------------+-------------+
|4869f7a5dfa277a7dca6462dcf3b52b2|249640.7             |1156         |
|7c67e1448b00f6e969d365cea6b010ab|239536.4399999998    |1364         |
|53243585a1d6dc2643021fd1853d8905|235856.68000000028   |410          |
|4a3ca9315b744ce9f8e9374361493884|235539.96000000057   |1987         |
|fa1c13f2614d7b5c4749cbc52fecda94|204084.7299999998    |586          |
|da8622b14eb17ae2831f4ac5b9dab84a|185192.31999999972   |1551         |
|7e93a43ef30c4f03f38b393420bc753a|182754.05000000005   |340          |
|1025f0e2d44d7041d6cf58b6550e0bfa|172860.69            |1428         |
|7a67c85e85bb2ce8582c35f2203ad736|162648.37999999998   |1171         |
|955fee9216a65b617aa5c0531780ce60|160602.67999999953   |1499         |
+--------------------------------+---------------------+-------------+
only s